# 🔬 Synthetic Data Generation on OVHcloud AI Notebooks (US)

This notebook orchestrates an **OVHcloud AI Notebook** (US-East, Virginia) to run the SDG pipeline at scale.

**What this does:**
1. Installs the `ovhai` CLI for US region
2. Authenticates with your OVHcloud US account
3. Launches a managed AI Notebook (JupyterLab + GPU)
4. Uploads the generation payload
5. Runs `sdg_pipeline.py` inside the OVH notebook
6. Pulls results back

**Prerequisites:**
- OVHcloud US Public Cloud project
- AI Training Operator + ObjectStore Operator permissions
- Valid payment method on the project

**Region:** US-East-VA (Vint Hill, Virginia) — `https://us-east-va.ai.cloud.ovh.us`

**Inference provider:** Uses OVH AI Endpoints OpenAI-compatible API (`oai.endpoints.kepler.ai.cloud.ovh.net/v1`) from inside the notebook.

In [ ]:
# === Cell 1: Install ovhai CLI (US region) ===
import os, sys, subprocess, json, textwrap, time
from pathlib import Path

OVH_REGION = 'us-east-va'
OVH_API_BASE = f'https://{OVH_REGION}.ai.cloud.ovh.us'
OVH_CLI_INSTALL = f'https://cli.{OVH_REGION}.ai.cloud.ovh.us/install.sh'

!curl -fsSL {OVH_CLI_INSTALL} | bash

# Add ovhai to PATH
os.environ['PATH'] = f"{os.environ.get('HOME','/root')}/bin:{os.environ['PATH']}"
!ovhai version

In [ ]:
# === Cell 2: Authenticate ===
# Run this interactively — it will open a browser or terminal prompt.
# Choose 'browser' method when prompted for easiest auth.

print('Run the cell below and follow the prompts.')
print('If using a restricted environment, choose terminal auth.')

# Uncomment to authenticate:
# !ovhai login

In [ ]:
# === Cell 3: API Token (for programmatic access) ===
import getpass

# Generate a token via OVH Control Panel:
#   AI & Machine Learning → AI Endpoints → API Keys → Create
# OR use ovhai CLI: ovhai token create --help

ovh_token = os.environ.get('OVH_AI_TOKEN', '')
if not ovh_token:
    ovh_token = getpass.getpass('OVHcloud AI Token: ')

ai_endpoint_key = os.environ.get('AI_ENDPOINT_API_KEY', '')
if not ai_endpoint_key:
    ai_endpoint_key = getpass.getpass('OVH AI Endpoints API Key (inference): ')

print(f'Token: ...{ovh_token[-8:]}')
print(f'Inference key: ...{ai_endpoint_key[-8:]}')

In [ ]:
# === Cell 4: Configuration ===
from pathlib import Path

# Generation parameters
NEMO_MODEL      = 'mistral@latest'        # OVH virtual model tag
NEMO_INTERVAL   = 2.0                    # safe interval for AI Endpoints
TARGET_COUNT    = 1000
MAX_ITERATIONS  = 5000
CLIP_VALIDITY   = 0.0

# Notebook spec
NB_NAME = 'pixelated-sdg'
NB_FLAVOR = 'ai1-1-gh'                  # 1xGPU H100 ; use 'cpu-a' for CPU-only
NB_ENV = 'python3.11-pytorch'           # pre-installed PyTorch + common deps

# Categories
CATEGORIES = [
    'somatic_therapy','attachment_disorders','personality_disorders',
    'dissociation','complicated_grief','eating_disorders',
    'ocd_intrusive_thoughts','narcissistic_abuse_recovery',
    'neurodivergent_mental_health','cultural_religious_contexts'
]

print(f'Notebook name: {NB_NAME}')
print(f'Flavor: {NB_FLAVOR}')
print(f'Environment: {NB_ENV}')
print(f'Categories: {len(CATEGORIES)}')

In [ ]:
# === Cell 5: Build the payload notebook (uploaded to OVH) ===
# We write a secondary .ipynb that will run INSIDE the OVH AI Notebook.

PAYLOAD_NB = Path('ovh_payload.ipynb')

payload = {
    'cells': [
        {'cell_type': 'markdown', 'metadata': {},
         'source': ['# OVH AI Notebook — SDG Payload\n','Run this inside the OVH notebook.']},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["import os, sys, subprocess\n","from pathlib import Path\n","# Clone repo\n","!git clone https://github.com/daggerstuff/pixelated.git /home/jovyan/pixelated\n","!cd /home/jovyan/pixelated && git submodule init && git submodule update\n","sys.path.insert(0, '/home/jovyan/pixelated/ai')\n","print('Repo ready.')"]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["# Install deps\n","!pip install -q uv\n","!uv sync --active 2>/dev/null || pip install -q -r /home/jovyan/pixelated/ai/requirements.txt\n","print('Env ready.')"]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["# Setup inference credentials\n","os.environ['AI_ENDPOINT_API_KEY'] = '{api_key}'\n","os.environ['NVIDIA_BASE_URL'] = 'https://oai.endpoints.kepler.ai.cloud.ovh.net/v1'\n","print('Credentials set.')".format(api_key=ai_endpoint_key)]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["# Generate batches\n","OUTPUT = Path('/home/jovyan/generated')\n","OUTPUT.mkdir(exist_ok=True)\n","categories = {cats}\n","print(f'Categories: {{len(categories)}}')".format(cats=CATEGORIES)]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["from training.sdg_pipeline import build_parser\n","from pathlib import Path\n","parser = build_parser()\n","\n","for cat in categories:\n","    out = OUTPUT / f'{cat}.jsonl'\n","    args = parser.parse_args([\n","        '--scenario','niche_category','--category',cat,\n","        '--target_count',str({target}),'--max_iterations',str({max_iter}),\n","        '--nemo_endpoint','https://oai.endpoints.kepler.ai.cloud.ovh.net/v1',\n","        '--nemo_api_key',os.environ['AI_ENDPOINT_API_KEY'],\n","        '--nemo_model','{model}','--nemo_timeout','60',\n","        '--nemo_min_call_interval',str({interval}),\n","        '--min_clinical_validity',str({clip}),\n","        '--output_path',str(out)])\n","    # run directly via module\n","    import training.sdg_pipeline as sp\n","    sp.run_sdg(args)\n","    print(f'  {cat}: {{sum(1 for _ in open(out))}} samples')".format(target='TARGET_COUNT',max_iter='MAX_ITERATIONS',model=NEMO_MODEL,interval=NEMO_INTERVAL,clip=CLIP_VALIDITY)]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["# DPO and Nightmare\n","for sc, name, t in [('dpo_preference_pairs','dpo_pairs',2000),('nightmare_fuel','nightmare_fuel',1000)]:\n","    out = OUTPUT / f'{name}.jsonl'\n","    args = parser.parse_args([\n","        '--scenario',sc,'--target_count',str(t),'--max_iterations','5000',\n","        '--nemo_endpoint','https://oai.endpoints.kepler.ai.cloud.ovh.net/v1',\n","        '--nemo_api_key',os.environ['AI_ENDPOINT_API_KEY'],\n","        '--nemo_model','{model}','--nemo_timeout','60',\n","        '--nemo_min_call_interval',str({interval}),\n","        '--min_clinical_validity',str({clip}),\n","        '--output_path',str(out)])\n","    sp.run_sdg(args)\n","    print(f'  {name}: {{sum(1 for _ in open(out))}} samples')".format(model=NEMO_MODEL,interval=NEMO_INTERVAL,clip=CLIP_VALIDITY)]},
        {'cell_type': 'code', 'execution_count': None, 'metadata': {},
         'outputs': [],
         'source': ["# Summary\n","total = 0\n","for f in OUTPUT.glob('*.jsonl'):\n","    n = sum(1 for _ in open(f))\n","    total += n\n","    print(f'{f.name:40s} {n:6d}')\n","print(f"Total: {total}")"]}
    ],
    'metadata': {'kernelspec': {'display_name':'Python 3','language':'python','name':'python3'},
                 'language_info': {'name':'python'}},
    'nbformat': 4, 'nbformat_minor': 4
}

PAYLOAD_NB.write_text(json.dumps(payload, indent=1))
print(f'Payload notebook written: {PAYLOAD_NB}')
print(f'Size: {PAYLOAD_NB.stat().st_size} bytes')

In [ ]:
# === Cell 6: Launch OVH AI Notebook ===
# NOTE: ovhai must be authenticated before this cell works (Cell 2).

import subprocess

launch_cmd = [
    'ovhai', 'notebook', 'run',
    '--name', NB_NAME,
    '--flavor', NB_FLAVOR,
    '--env', NB_ENV,
    '--token', ovh_token,
    '--region', OVH_REGION,
    '--volume', 'ovh-sdg-data@GRA:/home/jovyan/data:RW',  # optional: attach Object Storage
]

print(f'Launching {NB_NAME} on {OVH_REGION}...')
print('Command:', ' '.join(launch_cmd[:8]) + ' ...\n')

# Uncomment to actually launch:
# result = subprocess.run(launch_cmd, capture_output=True, text=True, env={**os.environ})
# print(result.stdout)
# print(result.stderr)

print("Launch command prepared. Uncomment the subprocess.run() to execute.")
print("")
print("Once launched, the notebook URL will appear in the ovhai output.")
print("Upload 'ovh_payload.ipynb' to the launched notebook and run it there.")

In [ ]:
# === Cell 7: Alternative — Run via ovhai CLI directly (no persistent notebook) ===
# If you prefer launching as a job instead of a persistent notebook:

job_cmd = [
    'ovhai', 'job', 'run',
    '--name', f'{NB_NAME}-job',
    '--flavor', NB_FLAVOR,
    '--env', NB_ENV,
    '--token', ovh_token,
    '--region', OVH_REGION,
    '--timeout', '24h',
    # Mount a git repo
    '--git-repo', 'https://github.com/daggerstuff/pixelated.git',
    '--git-ref', 'staging',
    # Run the SDG pipeline
    '--command', 'python -m training.sdg_pipeline ...',
]

print('Job launch command (for CLI copy-paste):')
print('\\')
print(' \\".join(job_cmd[:6]) + ' \\\')
print('   ...')

In [ ]:
# === Cell 8: Manual steps if not using ovhai CLI ===
from IPython.display import Markdown

steps = """
### Manual OVH AI Notebook Setup (US)

If the `ovhai` CLI doesn't work in your environment, use the OVH Control Panel:

1. **Go to**: [OVHcloud US Control Panel](https://us.ovhcloud.com) → Public Cloud → AI & ML → AI Notebooks
2. **Click**: Create a Notebook
3. **Region**: Select `US-EAST-VA` (Vint Hill, Virginia)
4. **Framework**: Choose `python3.11-pytorch` (or latest)
5. **Resources**: GPU H100 (`ai1-1-gh`) for fast inference
6. **Storage**: Attach Object Storage bucket `ovh-sdg-data` (read-write)
7. **Launch**: Wait for RUNNING status
8. **Upload**: `ovh_payload.ipynb` to the JupyterLab file manager
9. **Run**: Open `ovh_payload.ipynb` and execute all cells
10. **Download**: Generated files from `/home/jovyan/generated/*.jsonl`

### Inference credentials inside the notebook:

```python
os.environ['AI_ENDPOINT_API_KEY'] = '<your-inference-key>'
os.environ['NVIDIA_BASE_URL'] = 'https://oai.endpoints.kepler.ai.cloud.ovh.net/v1'
```
"""

display(Markdown(steps))

In [ ]:
# === Cell 9: Monitor / Clean up ===
# List running notebooks
print('Run this in terminal to list notebooks:')
print('  ovhai notebook list\n')

print('To stop a notebook:')
print(f'  ovhai notebook stop <uuid>\n')

print('To get notebook info:')
print(f'  ovhai notebook get <uuid>')